<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Notebooks/10_Dimensionality_Reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 10: Dimensionality Reduction & Unsupervised Learning

*Machine Learning Foundations — Codecademy Live Learning*

In this notebook we will:
- Understand **why** dimensionality reduction matters and when to use it
- Build intuition for **PCA** through visualizations and hands-on implementation
- Compare **PCA**, **t-SNE**, and **UMAP** for visualization
- Combine **PCA + K-Means** in a complete unsupervised learning pipeline
- Practice every coding pattern you'll need for **Project 3**

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import (load_digits, load_wine, make_blobs,
                               make_swiss_roll, fetch_california_housing)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print("All imports successful!")

---
# Section 1: Why Reduce Dimensions

Every feature in your dataset is a dimension in your feature space. When you have hundreds or thousands of features, three problems emerge: distance metrics break down, models overfit, and computation slows to a crawl. Dimensionality reduction addresses all three.

Let's see this concretely.

### The Curse of Dimensionality: A Quick Demonstration

In [ ]:
# How distance ratios degrade as dimensions increase
# In low dimensions, nearest and farthest neighbors are meaningfully different.
# In high dimensions, they converge.

dims = [2, 5, 10, 50, 100, 500, 1000]
ratios = []

for d in dims:
    points = np.random.randn(200, d)
    query = np.random.randn(1, d)
    dists = cdist(query, points, metric='euclidean').flatten()
    ratio = dists.max() / dists.min()
    ratios.append(ratio)

print(f"In 2D, farthest point is {ratios[0]:.1f}x farther than nearest")
print(f"In 1000D, farthest point is only {ratios[-1]:.1f}x farther than nearest")
print("\nWhen this ratio approaches 1, 'nearest neighbor' becomes meaningless.")

In [ ]:
# Visualize the distance ratio collapse
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dims, ratios, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Number of Dimensions')
ax.set_ylabel('Max Distance / Min Distance')
ax.set_title('Distance Ratio Collapses in High Dimensions')
ax.axhline(y=1.0, color='coral', linestyle='--', alpha=0.7, label='Ratio = 1 (all equidistant)')
ax.legend()
ax.set_xscale('log')
plt.tight_layout()
plt.show()

### Motivating Example: KNN on High-Dimensional Data

In [ ]:
# Load the digits dataset: 8x8 pixel images = 64 features
digits = load_digits()
X, y = digits.data, digits.target
print(f"Dataset shape: {X.shape} — {X.shape[1]} features (dimensions)")
print(f"Classes: {np.unique(y)}")
print(f"Samples: {X.shape[0]}")

In [ ]:
# Train KNN on the full 64-dimensional data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_full = KNeighborsClassifier(n_neighbors=5)
knn_full.fit(X_train_scaled, y_train)
acc_full = knn_full.score(X_test_scaled, y_test)
print(f"KNN accuracy on all {X.shape[1]} features: {acc_full:.4f}")

In [ ]:
# Now reduce to 20 dimensions with PCA and try again
pca_20 = PCA(n_components=20, random_state=42)
X_train_pca = pca_20.fit_transform(X_train_scaled)
X_test_pca = pca_20.transform(X_test_scaled)

knn_pca = KNeighborsClassifier(n_neighbors=5)
knn_pca.fit(X_train_pca, y_train)
acc_pca = knn_pca.score(X_test_pca, y_test)
print(f"KNN accuracy on {20} PCA components: {acc_pca:.4f}")
print(f"Variance retained: {pca_20.explained_variance_ratio_.sum():.1%}")
print(f"\nWe reduced dimensions by {100*(1 - 20/X.shape[1]):.0f}% with minimal accuracy change.")

Let's see that PCA with 20 components preserves much more of the information than just truncating the data from 64 columns to 20.

In [ ]:
# Just take the first 20 columns of data
X_train_scaled_20 = X_train_scaled[:, :20]
X_test_scaled_20 = X_test_scaled[:, :20]
X_train_scaled_20[0]

In [ ]:
knn_trunc = KNeighborsClassifier(n_neighbors=5)
knn_trunc.fit(X_train_scaled_20, y_train)
acc_pca = knn_trunc.score(X_test_scaled_20, y_test)
print(f"KNN accuracy on {20} non-PCA components: {acc_pca:.4f}")


---
# Section 2: PCA Intuition

PCA finds new axes (principal components) that align with the directions of maximum variance in your data. Think of it as rotating your coordinate system so that the first axis captures the most spread, the second captures the next most, and so on.

The key insight: **variance = information**. Directions with high variance carry signal; directions with low variance are mostly noise.

### 2D → 1D Projection: Seeing PCA in Action

In [ ]:
# Create correlated 2D data and fit PCA
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]
X_2d = np.random.multivariate_normal(mean, cov, 200)

In [ ]:
# Visualize original 2D data
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4
           , color='steelblue', s=20)

In [ ]:
pca_2d = PCA(n_components=2)
pca_2d.fit(X_2d)
X_transformed = pca_2d.transform(X_2d)

print(f"PC1 captures {pca_2d.explained_variance_ratio_[0]:.1%} of the total variance")
print(f"PC2 captures {pca_2d.explained_variance_ratio_[1]:.1%} of the total variance")
print(f"\nBy keeping only PC1, we compress 2D → 1D while retaining {pca_2d.explained_variance_ratio_[0]:.1%} of the information")

In [ ]:
# Visualize: original data, rotated, and projected to 1D
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Original data with PC directions
ax = axes[0]
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4, color='steelblue', s=20)
origin = pca_2d.mean_
for i, (comp, var) in enumerate(zip(pca_2d.components_, pca_2d.explained_variance_)):
    scale = np.sqrt(var) * 2
    ax.annotate('', xy=origin + comp * scale, xytext=origin,
                arrowprops=dict(arrowstyle='->', color='coral' if i == 0 else 'forestgreen', lw=3))
    ax.text(origin[0] + comp[0] * scale * 1.15, origin[1] + comp[1] * scale * 1.15,
            f'PC{i+1} ({pca_2d.explained_variance_ratio_[i]:.1%})',
            fontsize=12, fontweight='bold', color='coral' if i == 0 else 'forestgreen')
ax.set_xlabel('X₁'); ax.set_ylabel('X₂')
ax.set_title('Original Data with Principal Components')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# Plot 2: In PCA coordinates (rotated)
ax = axes[1]
ax.scatter(X_transformed[:, 0], X_transformed[:, 1], alpha=0.4, color='steelblue', s=20)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Data in PCA Coordinates (Rotated)')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

# Plot 3: Projected onto PC1 only
ax = axes[2]
ax.scatter(X_transformed[:, 0], np.zeros_like(X_transformed[:, 0]), alpha=0.4, color='coral', s=20)
ax.set_xlabel('PC1')
ax.set_title(f'Projected onto PC1 Only ({pca_2d.explained_variance_ratio_[0]:.1%} of variance)')
ax.set_yticks([]); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

### What Just Happened?

1. **Left plot:** The original data with two correlated features. The red arrow (PC1) points in the direction of maximum spread. The green arrow (PC2) is perpendicular and captures the remaining spread.

2. **Middle plot:** Same data after rotating to PCA coordinates. Notice the axes are now uncorrelated (no more diagonal elongation).

3. **Right plot:** We dropped PC2 and kept only PC1. We went from 2D to 1D, losing some information but retaining the majority of the variance.

This is exactly what PCA does in higher dimensions — it finds the directions of maximum spread and projects your data onto them.

### Why Standardization Matters

In [ ]:
# Create data where feature 2 has much larger scale
np.random.seed(42)
X_unscaled = np.column_stack([
    np.random.randn(200) * 1,       # Feature 1: small scale
    np.random.randn(200) * 100      # Feature 2: large scale (but same information content)
])

# PCA WITHOUT scaling
pca_no_scale = PCA(n_components=2)
pca_no_scale.fit(X_unscaled)

# PCA WITH scaling
scaler = StandardScaler()
X_scaled_demo = scaler.fit_transform(X_unscaled)
pca_with_scale = PCA(n_components=2)
pca_with_scale.fit(X_scaled_demo)

print("WITHOUT standardization:")
print(f"  PC1: {pca_no_scale.explained_variance_ratio_[0]:.4f}  PC2: {pca_no_scale.explained_variance_ratio_[1]:.4f}")
print("  → PC1 is completely dominated by the large-scale feature")
print()
print("WITH standardization:")
print(f"  PC1: {pca_with_scale.explained_variance_ratio_[0]:.4f}  PC2: {pca_with_scale.explained_variance_ratio_[1]:.4f}")
print("  → Both features contribute roughly equally")
print()
print("Always standardize before PCA unless all features are already on the same scale.")

In [ ]:
# Side-by-side visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pca_obj, title in zip(axes,
    [pca_no_scale, pca_with_scale],
    ['PCA WITHOUT Standardization', 'PCA WITH Standardization']):
    ax.bar(['PC1', 'PC2'], pca_obj.explained_variance_ratio_, color=['coral', 'steelblue'])
    ax.set_title(title)
    ax.set_ylabel('Explained Variance Ratio')
    ax.set_ylim(0, 1.05)
    for i, v in enumerate(pca_obj.explained_variance_ratio_):
        ax.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Exercise 1: PCA on the Wine Dataset

Apply PCA to the Wine dataset from sklearn. This dataset has 13 chemical measurements from three types of wine.

**Tasks:**

1. Load the wine dataset and standardize the features
2. Fit PCA with all components and create a scree plot (bar chart of explained variance per component)
3. Create a cumulative explained variance plot and determine how many components are needed to retain 90% of the variance
4. Transform the data to 2D and create a scatter plot colored by wine class
5. Examine the loadings for PC1 and PC2 — which original features contribute most?


### Task 1: Load and Standardize

In [ ]:
# Load the wine dataset with load_wine()
# Standardize the features using StandardScaler
# Print the shape and feature names

# Your code here


In [ ]:
#@title Click to reveal solution
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = wine.feature_names

scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine)
print(f"Wine dataset: {X_wine.shape[0]} samples, {X_wine.shape[1]} features")
print(f"Features: {list(feature_names)}")
print(f"Classes: {list(wine.target_names)}")

### Task 2: Scree Plot

In [ ]:
# Fit PCA with ALL components (don't specify n_components)
# Create a bar chart of explained_variance_ratio_ for each component

# Your code here


In [ ]:
#@title Click to reveal solution
pca_wine = PCA(random_state=42)
pca_wine.fit(X_wine_scaled)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, len(pca_wine.explained_variance_ratio_) + 1),
       pca_wine.explained_variance_ratio_, color='steelblue', alpha=0.8)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('Scree Plot: Wine Dataset')
ax.set_xticks(range(1, 14))
plt.tight_layout()
plt.show()

### Task 3: Cumulative Variance — How Many Components?

In [ ]:
# Compute cumulative explained variance using np.cumsum()
# Plot it as a line chart
# Add a horizontal line at 0.90 for the 90% threshold
# Print how many components are needed to hit 90%

# Your code here


In [ ]:
#@title Click to reveal solution
cumulative = np.cumsum(pca_wine.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumulative) + 1), cumulative, 'o-', color='coral', linewidth=2, markersize=6)
ax.axhline(y=0.90, color='gray', linestyle='--', alpha=0.7, label='90% threshold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('Cumulative Explained Variance: Wine Dataset')
ax.set_xticks(range(1, 14))
ax.legend()
plt.tight_layout()
plt.show()

n_components_90 = np.argmax(cumulative >= 0.90) + 1
print(f"Components needed for 90% variance: {n_components_90}")
print(f"That's {n_components_90} components instead of {X_wine.shape[1]} original features")

### Task 4: 2D Visualization

In [ ]:
# Fit PCA with n_components=2
# Transform the scaled data
# Create a scatter plot colored by wine class (y_wine)
# Hint: loop over classes and plot each one separately to get a legend

# Your code here


In [ ]:
#@title Click to reveal solution
pca_2 = PCA(n_components=2, random_state=42)
X_wine_2d = pca_2.fit_transform(X_wine_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['steelblue', 'coral', 'forestgreen']
for i, name in enumerate(wine.target_names):
    mask = y_wine == i
    ax.scatter(X_wine_2d[mask, 0], X_wine_2d[mask, 1], c=colors[i],
               label=name, alpha=0.7, s=50, edgecolors='white', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca_2.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Wine Dataset: First 2 Principal Components')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Task 5: Examine Loadings

In [ ]:
# The loadings are in pca_2.components_ (shape: n_components x n_features)
# For PC1 and PC2, find the top 5 features by absolute loading value
# Hint: create a pd.Series with feature_names as the index, then use .abs().sort_values()

# Your code here


In [ ]:
#@title Click to reveal solution
print("PC1 Loadings (top 5 by magnitude):")
loadings_pc1 = pd.Series(pca_2.components_[0], index=feature_names)
top_pc1 = loadings_pc1.abs().sort_values(ascending=False).head(5)
for feat in top_pc1.index:
    print(f"  {feat:>25s}: {loadings_pc1[feat]:+.3f}")

print("\nPC2 Loadings (top 5 by magnitude):")
loadings_pc2 = pd.Series(pca_2.components_[1], index=feature_names)
top_pc2 = loadings_pc2.abs().sort_values(ascending=False).head(5)
for feat in top_pc2.index:
    print(f"  {feat:>25s}: {loadings_pc2[feat]:+.3f}")

---
# Section 3: PCA in Practice

Now that we have the intuition, let's look at the practical mechanics: the sklearn API, pipelines, and the critical rule about fitting on training data only.

### The PCA API in scikit-learn

In [ ]:
# Quick reference for the PCA API
digits = load_digits()
X, y = digits.data, digits.target


In [ ]:
# Step 1: Always standardize first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)



In [ ]:
# Step 2: Fit PCA — you can specify a number OR a variance threshold
pca = PCA(n_components=0.95, random_state=42)  # Keep 95% of variance
X_pca = pca.fit_transform(X_scaled)

print("Key PCA attributes:")
print(f"  n_components_ (chosen):    {pca.n_components_}")
print(f"  explained_variance_ratio_: {pca.explained_variance_ratio_[:5].round(3)}... (first 5)")
print(f"  Total variance retained:   {sum(pca.explained_variance_ratio_):.1%}")
print(f"  components_ shape:         {pca.components_.shape}  (n_components x n_features)")
print(f"\nOriginal shape: {X_scaled.shape} → Reduced shape: {X_pca.shape}")

### Avoiding Data Leakage: Fit on Train Only

In [ ]:
# CORRECT: fit PCA on training data only, transform both
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)       # transform, NOT fit_transform

pca = PCA(n_components=20, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)       # transform, NOT fit_transform

print("Correct workflow: fit on train, transform both")
print(f"  Train: {X_train.shape} → {X_train_pca.shape}")
print(f"  Test:  {X_test.shape} → {X_test_pca.shape}")

### PCA in a Pipeline (The Clean Way)

In [ ]:
# A pipeline bundles preprocessing + model, preventing leakage automatically
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

pipe.fit(X_train, y_train)
print(f"Pipeline accuracy: {pipe.score(X_test, y_test):.4f}")


In [ ]:
# You can access individual steps
pca_step = pipe.named_steps['pca']
print(f"Variance retained: {sum(pca_step.explained_variance_ratio_):.1%}")

### Reconstruction: Seeing What PCA Keeps and Discards

In [ ]:
# Load Fashion MNIST for a clearer reconstruction demo
# Fashion MNIST has 28x28 images = 784 dimensions
from sklearn.datasets import fetch_openml

print("Loading Fashion MNIST (this may take a moment)...")
fmnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X_fashion = fmnist.data[:5000].astype(float)  # use a subset for speed
y_fashion = fmnist.target[:5000].astype(int)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

scaler_f = StandardScaler()
X_fashion_scaled = scaler_f.fit_transform(X_fashion)
print(f"Loaded: {X_fashion.shape[0]} images, {X_fashion.shape[1]} pixels each")

In [ ]:
# Reconstruct a sample image using different numbers of PCA components
sample_idx = 0
original = X_fashion_scaled[sample_idx]
label = class_names[y_fashion[sample_idx]]

n_components_list = [5, 20, 50, 100, 200, 784]
reconstructions = {}

for n in n_components_list:
    if n >= X_fashion_scaled.shape[1]:
        reconstructions[n] = scaler_f.inverse_transform(original.reshape(1, -1)).flatten()
    else:
        pca_temp = PCA(n_components=n, random_state=42)
        pca_temp.fit(X_fashion_scaled)
        compressed = pca_temp.transform(original.reshape(1, -1))
        reconstructed_scaled = pca_temp.inverse_transform(compressed)
        reconstructions[n] = scaler_f.inverse_transform(reconstructed_scaled).flatten()

print(f"Sample image: {label}")
print(f"Reconstructing with {len(n_components_list)} different component counts...")

In [ ]:
# Visualize the reconstruction quality
fig, axes = plt.subplots(1, len(n_components_list), figsize=(18, 3))

for ax, n in zip(axes, n_components_list):
    img = reconstructions[n].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    if n >= 784:
        ax.set_title(f'Original\n(784 dims)')
    else:
        pca_temp = PCA(n_components=n, random_state=42).fit(X_fashion_scaled)
        variance = sum(pca_temp.explained_variance_ratio_)
        ax.set_title(f'{n} PCs\n({variance:.0%} var)')
    ax.axis('off')

plt.suptitle(f'Reconstructing a "{label}" from Different Numbers of Components', fontsize=13)
plt.tight_layout()
plt.show()

print("With just 50 components (6% of the original 784 dimensions),")
print("the image is clearly recognizable. That's the power of PCA as compression.")

---
# Section 4: t-SNE and UMAP for Visualization

PCA is linear — it can only find straight-line directions of variance. When data has nonlinear structure (curves, clusters within clusters), PCA projections can look like an undifferentiated blob. That's where t-SNE and UMAP come in.

**Critical distinction:** t-SNE and UMAP are **visualization tools**, not general-purpose preprocessing. Use PCA when you need to reduce dimensions for modeling. Use t-SNE/UMAP when you need a human-readable 2D picture of your data.

### When PCA Isn't Enough: Nonlinear Structure

In [ ]:
# The Swiss Roll: a classic example of nonlinear structure
X_swiss, color = make_swiss_roll(n_samples=1500, noise=0.5, random_state=42)

# PCA projection
pca_swiss = PCA(n_components=2)
X_swiss_pca = pca_swiss.fit_transform(X_swiss)

print("The Swiss Roll is a 2D surface curled up in 3D space.")
print("PCA can only project linearly — it can't 'unroll' the structure.")

In [ ]:
# Visualize 3D data, PCA projection, and ideal unrolling
fig = plt.figure(figsize=(16, 5))

ax = fig.add_subplot(131, projection='3d')
ax.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=color, cmap='Spectral', s=10, alpha=0.7)
ax.set_title('Swiss Roll in 3D')

ax2 = fig.add_subplot(132)
ax2.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], c=color, cmap='Spectral', s=10, alpha=0.7)
ax2.set_title('PCA Projection (2D)')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')

ax3 = fig.add_subplot(133)
ax3.scatter(X_swiss[:, 0], color, c=color, cmap='Spectral', s=10, alpha=0.7)
ax3.set_title('What We Want: Unrolled')
ax3.set_xlabel('X'); ax3.set_ylabel('Position on roll')

plt.tight_layout()
plt.show()

In [ ]:
tsne_swiss = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_swiss_tsne = tsne_swiss.fit_transform(X_swiss)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_swiss_tsne[:, 0], X_swiss_tsne[:, 1], c=color, cmap='Spectral', s=20, alpha=0.7)
ax.set_title('t-SNE Projection of Swiss Roll (2D)')
ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
plt.colorbar(scatter, ax=ax, label='Position on roll')
plt.tight_layout()
plt.show()

For the swiss roll, t-SNE preserve the along-the-surface distances rather than the straight-line-through-3D-space distances.

### t-SNE: Preserving Local Neighborhoods

In [ ]:
# Compare PCA and t-SNE on the digits dataset
digits = load_digits()
X_digits, y_digits = digits.data, digits.target

scaler_d = StandardScaler()
X_digits_scaled = scaler_d.fit_transform(X_digits)

# PCA to 2D
pca_digits = PCA(n_components=2, random_state=42)
X_pca_digits = pca_digits.fit_transform(X_digits_scaled)

# t-SNE to 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_digits_scaled)

print("PCA and t-SNE both reduce to 2D, but they preserve different things.")
print("PCA preserves global variance; t-SNE preserves local neighborhoods.")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_proj, title in zip(axes,
                              [X_pca_digits, X_tsne],
                              ['PCA', 't-SNE (perplexity=30)']):
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y_digits, cmap='tab10',
                         s=10, alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')

plt.colorbar(scatter, ax=axes[1], label='Digit Class', ticks=range(10))
plt.suptitle('Digits Dataset: PCA vs t-SNE', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### t-SNE Hyperparameter Sensitivity: Why You Should Be Careful

In [ ]:
# The SAME data with different perplexity values produces VERY different plots
perplexities = [5, 15, 30, 100]
tsne_results = {}

for perp in perplexities:
    tsne_temp = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    tsne_results[perp] = tsne_temp.fit_transform(X_digits_scaled)

print("Generated t-SNE projections for perplexity = 5, 15, 30, 100")

In [ ]:
# Visualize all four
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, perp in zip(axes, perplexities):
    ax.scatter(tsne_results[perp][:, 0], tsne_results[perp][:, 1],
               c=y_digits, cmap='tab10', s=8, alpha=0.6)
    ax.set_title(f'Perplexity = {perp}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Same Data, Different Perplexity → Different Plots', fontsize=14)
plt.tight_layout()
plt.show()

print("Key takeaways:")
print("  - Low perplexity (5): tight, fragmented clusters")
print("  - High perplexity (100): broader, merged groupings")
print("  - Always try multiple values before drawing conclusions")
print("  - Distances BETWEEN clusters are NOT meaningful in t-SNE")

### UMAP: Faster, Preserves More Global Structure

In [ ]:
# UMAP — graceful fallback if not installed
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X_digits_scaled)
    has_umap = True
    print("UMAP projection complete.")
except ImportError:
    has_umap = False
    print("UMAP is not installed. To install: !pip install umap-learn")
    print("Then restart runtime and re-run. UMAP is optional for this notebook.")

In [ ]:
# Compare all three (or just PCA and t-SNE if no UMAP)
if has_umap:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    data_list = [X_pca_digits, X_tsne, X_umap]
    titles = ['PCA', 't-SNE', 'UMAP']
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    data_list = [X_pca_digits, X_tsne]
    titles = ['PCA', 't-SNE']

for ax, X_proj, title in zip(axes, data_list, titles):
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y_digits, cmap='tab10',
                         s=10, alpha=0.7)
    ax.set_title(title, fontsize=14)
    ax.set_xticks([]); ax.set_yticks([])

plt.colorbar(scatter, ax=axes[-1], label='Digit', ticks=range(10))
plt.suptitle('Comparison of Dimensionality Reduction Methods', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### What You Can and Can't Conclude from t-SNE / UMAP

**Reliable interpretations:**
- Points that cluster together are genuinely similar in the high-dimensional space
- Distinct, well-separated groups suggest real structure in the data

**Unreliable interpretations:**
- Distances *between* clusters (especially in t-SNE) don't reflect true distances
- Cluster *sizes* in the plot don't reflect actual density
- Apparent structure can be an artifact of hyperparameter choices

**Rule of thumb:** If you see clear clusters in t-SNE/UMAP, they're probably real. But always validate with other methods or domain knowledge before making claims.

---
## Exercise 2: Comparing Visualization Methods on the Wine Dataset

**Tasks:**

1. Apply t-SNE to the standardized wine data and plot the result, colored by wine class
2. Try t-SNE with perplexity values of 5, 15, 30, and 50 — plot all four side by side
3. Which perplexity gives the clearest separation of the three wine classes?
4. Compare the PCA plot (from Exercise 1) to the best t-SNE plot — what does each reveal that the other doesn't?
5. (Optional) If UMAP is installed, create a UMAP visualization and compare all three methods

### Task 1: t-SNE with Multiple Perplexity Values

In [ ]:
# Apply t-SNE to X_wine_scaled (from Exercise 1) with perplexity = 5, 15, 30, 50
# Plot all four side by side, colored by y_wine
# Hint: follow the same pattern as the digits perplexity demo above

# Your code here


In [ ]:
#@title Click to reveal solution
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
perplexities = [5, 15, 30, 50]
colors = ['steelblue', 'coral', 'forestgreen']
best_tsne_wine = None

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_proj = tsne.fit_transform(X_wine_scaled)
    for i, name in enumerate(wine.target_names):
        mask = y_wine == i
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1], c=colors[i], label=name, s=30, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}')
    ax.set_xticks([]); ax.set_yticks([])
    if perp == 30:
        best_tsne_wine = X_proj
        ax.legend(fontsize=8)

plt.suptitle('t-SNE on Wine Dataset: Effect of Perplexity', fontsize=14)
plt.tight_layout()
plt.show()

### Task 2: Compare PCA vs t-SNE

In [ ]:
# Create a side-by-side plot: PCA (from Exercise 1) vs your best t-SNE result
# Which method shows clearer separation? What does each reveal?

# Your code here


In [ ]:
#@title Click to reveal solution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, X_proj, title in zip(axes, [X_wine_2d, best_tsne_wine], ['PCA', 't-SNE (perplexity=30)']):
    for i, name in enumerate(wine.target_names):
        mask = y_wine == i
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1], c=colors[i], label=name, s=40, alpha=0.7)
    ax.set_title(title, fontsize=13)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('PCA vs t-SNE: Wine Dataset', fontsize=14)
plt.tight_layout()
plt.show()

print("For the Wine dataset, PCA already shows good separation because")
print("the class structure is fairly linear in this feature space.")
print("t-SNE gives tighter local clusters but the global layout may differ between runs.")

---
# Section 5: Combining PCA with Clustering

This is where everything comes together. In practice, you'll often want to:
1. Reduce dimensions with PCA to remove noise and speed up clustering
2. Run K-Means in the reduced space
3. Visualize the clusters in 2D PCA space
4. Interpret the clusters by mapping back to original features

This is exactly the workflow you'll use in Project 3.

### Step 1: Standardize and Apply PCA

In [ ]:
# We'll use the Wine dataset for a complete walkthrough
wine = load_wine()
X_wine = wine.data
feature_names = wine.feature_names

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_wine)

# PCA — retain 95% variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Standardized: {X_scaled.shape}")
print(f"After PCA: {X_pca.shape} ({sum(pca.explained_variance_ratio_):.1%} variance retained)")

### Step 2: Cluster in Reduced Space

In [ ]:
# K-Means in PCA-reduced space
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_pca)

print(f"K-Means with k=3")
print(f"Cluster sizes: {np.bincount(cluster_labels)}")
print(f"Silhouette score: {silhouette_score(X_pca, cluster_labels):.3f}")

### Step 3: Visualize Clusters in 2D PCA Space

In [ ]:
# Plot clusters with centroids
# Since we clustered in PCA space, the centroids are already in PCA coordinates
centers_pca = kmeans.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_cluster = ['steelblue', 'coral', 'forestgreen']

# Left: K-Means clusters
ax = axes[0]
for i in range(3):
    mask = cluster_labels == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_cluster[i],
               label=f'Cluster {i}', alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1], c='black', marker='X',
           s=200, edgecolors='white', linewidth=2, zorder=5, label='Centroids')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('K-Means Clusters (k=3)')
ax.legend(); ax.grid(True, alpha=0.3)

# Right: True wine classes for comparison
ax = axes[1]
y_wine = wine.target
for i, name in enumerate(wine.target_names):
    mask = y_wine == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_cluster[i],
               label=name, alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('True Wine Classes')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Discovered Clusters vs True Classes', fontsize=14)
plt.tight_layout()
plt.show()

### Step 4: Interpret Clusters via Original Features

In [ ]:
# Cluster centroids in PCA space are abstract.
# To interpret, we map back to original feature space using inverse_transform.

# PCA inverse_transform gives us scaled features; scaler inverse_transform gives original scale
centers_original_scaled = pca.inverse_transform(kmeans.cluster_centers_)
centers_original = scaler.inverse_transform(centers_original_scaled)

profile = pd.DataFrame(centers_original, columns=feature_names)
profile.index = [f'Cluster {i}' for i in range(3)]

print("Cluster Profiles (mean feature values in original scale):")
print(profile.round(2).to_string())

In [ ]:
# An alternative (and often simpler) approach: compute group means directly
df_wine = pd.DataFrame(X_wine, columns=feature_names)
df_wine['cluster'] = cluster_labels

cluster_means = df_wine.groupby('cluster')[list(feature_names)].mean()
print("Cluster means computed directly from original data:")
print(cluster_means.round(2).to_string())

In [ ]:
# Visualize as a heatmap (use standardized values so features are comparable)
cluster_means_scaled = pd.DataFrame(X_scaled, columns=feature_names)
cluster_means_scaled['cluster'] = cluster_labels
cluster_means_scaled = cluster_means_scaled.groupby('cluster').mean()

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(cluster_means_scaled.values, cmap='RdBu_r', aspect='auto', vmin=-1.5, vmax=1.5)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels([f'Cluster {i}' for i in range(3)])
plt.colorbar(im, ax=ax, label='Standardized Mean Value')
ax.set_title('Cluster Profiles: Mean Standardized Feature Values')

for i in range(3):
    for j in range(len(feature_names)):
        val = cluster_means_scaled.values[i, j]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8,
                color='white' if abs(val) > 0.8 else 'black')
plt.tight_layout()
plt.show()

print("\nDistinguishing characteristics:")
for i in range(3):
    row = cluster_means_scaled.iloc[i]
    high = row.nlargest(2).index.tolist()
    low = row.nsmallest(2).index.tolist()
    print(f"  Cluster {i}: high {high[0]}, {high[1]}; low {low[0]}, {low[1]}")

### Comparing: With and Without PCA

In [ ]:
# Does PCA actually help clustering here?
kmeans_original = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_original = kmeans_original.fit_predict(X_scaled)

sil_original = silhouette_score(X_scaled, labels_original)
sil_pca = silhouette_score(X_pca, cluster_labels)

print("Silhouette Score Comparison:")
print(f"  K-Means on original {X_scaled.shape[1]} features: {sil_original:.3f}")
print(f"  K-Means on {X_pca.shape[1]} PCA components:     {sil_pca:.3f}")
print(f"\nPCA isn't always better for clustering; it depends on the data.")
print(f"The key benefit is often visualization and speed, not always cluster quality.")

### Loadings Heatmap: What Do the PCA Axes Mean?

In [ ]:
# Loadings show how each original feature maps to each PC
# This helps you interpret the axes in the cluster visualization

loadings = pd.DataFrame(
    pca.components_[:3],
    columns=feature_names,
    index=[f'PC{i+1} ({pca.explained_variance_ratio_[i]:.1%})' for i in range(3)]
)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(loadings.values, cmap='RdBu_r', aspect='auto', vmin=-0.6, vmax=0.6)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_yticks(range(3))
ax.set_yticklabels(loadings.index)
plt.colorbar(im, ax=ax, label='Loading Weight')
ax.set_title('PCA Loadings: How Original Features Map to Principal Components')

for i in range(3):
    for j in range(len(feature_names)):
        val = loadings.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(val) > 0.35 else 'black')
plt.tight_layout()
plt.show()

---
## Exercise 3: PCA + Clustering on the California Housing Dataset

Apply the full PCA + K-Means pipeline to the California Housing dataset (8 features describing census block groups in California).

**Tasks:**

1. Load the California Housing dataset and standardize the features
2. Fit PCA and create a scree plot — how many components capture 90% of the variance?
3. Apply K-Means with PCA-reduced features. Use the elbow method and silhouette scores to choose k (try k=2 through k=8)
4. Visualize the clusters in 2D PCA space with centroids marked
5. Create a cluster profile: compute mean original feature values per cluster and identify what distinguishes each cluster

### Task 1: Load, Standardize, and Apply PCA

In [ ]:
# Load with fetch_california_housing()
# Standardize with StandardScaler
# Fit PCA and create a scree plot
# How many components capture 90% of the variance?

# Your code here


In [ ]:
#@title Click to reveal solution
housing = fetch_california_housing()
X_housing = housing.data
feature_names_h = housing.feature_names

scaler_h = StandardScaler()
X_h_scaled = scaler_h.fit_transform(X_housing)

pca_h = PCA(random_state=42)
pca_h.fit(X_h_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, 9), pca_h.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

cumvar = np.cumsum(pca_h.explained_variance_ratio_)
axes[1].plot(range(1, 9), cumvar, 'o-', color='coral', linewidth=2)
axes[1].axhline(y=0.90, color='gray', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance'); axes[1].legend()
plt.tight_layout()
plt.show()

n_comp_90 = np.argmax(cumvar >= 0.90) + 1
print(f"Components for 90% variance: {n_comp_90}")

### Task 2: Choose k with Elbow and Silhouette

In [ ]:
# Apply PCA with the number of components from Task 1
# Run K-Means for k = 2 through 8
# Plot inertia (elbow) and silhouette scores
# Hint: California Housing is large, so use a random subsample of ~5000 points for speed
#   sample_idx = np.random.choice(len(X_pca), size=5000, replace=False)
#   X_sample = X_pca[sample_idx]

# Your code here


In [ ]:
#@title Click to reveal solution
pca_h_reduced = PCA(n_components=n_comp_90, random_state=42)
X_h_pca = pca_h_reduced.fit_transform(X_h_scaled)

np.random.seed(42)
sample_idx = np.random.choice(len(X_h_pca), size=5000, replace=False)
X_h_sample = X_h_pca[sample_idx]

ks = range(2, 9)
inertias = []
sil_scores = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_h_sample)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_h_sample, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(ks), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
axes[1].plot(list(ks), sil_scores, 'o-', color='coral', linewidth=2)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Analysis')
plt.tight_layout()
plt.show()

best_k = list(ks)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k}")

### Task 3: Visualize and Interpret Clusters

In [ ]:
# Fit the final K-Means model with your chosen k
# Visualize clusters in 2D PCA space (PC1 vs PC2) with centroids
# Create a cluster profile by computing mean feature values (original scale) per cluster
# Hint: use scaler.inverse_transform(pca.inverse_transform(centers)) to get original-scale centroids

# Your code here


In [ ]:
#@title Click to reveal solution
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels_final = km_final.fit_predict(X_h_sample)

# Visualize
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_h_sample[:, 0], X_h_sample[:, 1], c=labels_final,
                     cmap='tab10', s=10, alpha=0.5)
centers = km_final.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200,
           edgecolors='white', linewidth=2, zorder=5)
ax.set_xlabel(f'PC1 ({pca_h_reduced.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_h_reduced.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'California Housing: K-Means Clusters (k={best_k}) in PCA Space')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

# Cluster profiles
centers_scaled = pca_h_reduced.inverse_transform(km_final.cluster_centers_)
centers_original = scaler_h.inverse_transform(centers_scaled)

profile_h = pd.DataFrame(centers_original, columns=feature_names_h)
profile_h.index = [f'Cluster {i}' for i in range(best_k)]
print("Cluster Profiles (original scale):")
print(profile_h.round(2).to_string())

---
## Exercise 4: Project 3 Warmup — Full Unsupervised Pipeline

This exercise walks through the exact workflow you'll use in **Project 3** (Spotify Vibe Clustering), but on a synthetic dataset. Treat this as a dress rehearsal for the project.

We'll use the `make_blobs` synthetic dataset to keep the focus on the pipeline, not the data.

**Tasks:**

1. Generate a synthetic dataset with 5 clusters and 10 features. Standardize the features.
2. Apply PCA. Create a scree plot and choose the number of components.
3. Run K-Means with the elbow method and silhouette scores to choose k.
4. Fit the final K-Means model. Add cluster labels to a DataFrame.
5. Visualize clusters in 2D PCA space, including centroids.
6. Create a cluster profile: use `scaler.inverse_transform()` to get feature means in the original scale, and display as a heatmap.
7. Display 3 sample rows from each cluster.

### Task 1: Generate and Standardize Data

In [ ]:
# Use make_blobs to create 1000 samples, 10 features, 5 centers
# Wrap it in a DataFrame with column names 'feature_0' through 'feature_9'
# Standardize the features

# Your code here


In [ ]:
#@title Click to reveal solution
X_blob, y_blob = make_blobs(n_samples=1000, n_features=10, centers=5,
                             cluster_std=1.5, random_state=42)
feature_cols = [f'feature_{i}' for i in range(10)]
df = pd.DataFrame(X_blob, columns=feature_cols)

scaler_b = StandardScaler()
X_blob_scaled = scaler_b.fit_transform(X_blob)
print(f"Data: {X_blob.shape[0]} samples, {X_blob.shape[1]} features")

### Task 2: PCA Scree Plot

In [ ]:
# Fit PCA with all components on the scaled data
# Create a scree plot and cumulative variance plot
# How many components for 95% variance?

# Your code here


In [ ]:
#@title Click to reveal solution
pca_blob = PCA(random_state=42)
pca_blob.fit(X_blob_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, 11), pca_blob.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

cumvar_b = np.cumsum(pca_blob.explained_variance_ratio_)
axes[1].plot(range(1, 11), cumvar_b, 'o-', color='coral', linewidth=2)
axes[1].axhline(y=0.95, color='gray', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Explained Variance'); axes[1].legend()
plt.tight_layout()
plt.show()

n_comp_b = np.argmax(cumvar_b >= 0.95) + 1
print(f"Components for 95% variance: {n_comp_b}")

### Task 3: Elbow and Silhouette to Choose k

In [ ]:
# Apply PCA with your chosen number of components
# Run K-Means for k = 2 through 10
# Plot inertia and silhouette scores
# Which k looks best?

# Your code here


In [ ]:
#@title Click to reveal solution
pca_b_reduced = PCA(n_components=n_comp_b, random_state=42)
X_pca_blob = pca_b_reduced.fit_transform(X_blob_scaled)

ks = range(2, 11)
inertias_b = []
sils_b = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labs = km.fit_predict(X_pca_blob)
    inertias_b.append(km.inertia_)
    sils_b.append(silhouette_score(X_pca_blob, labs))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(ks), inertias_b, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
axes[1].plot(list(ks), sils_b, 'o-', color='coral', linewidth=2)
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Score')
plt.tight_layout()
plt.show()

best_k_b = list(ks)[np.argmax(sils_b)]
print(f"Best k by silhouette: {best_k_b}")

### Task 4: Fit Final Model and Add Labels to DataFrame

In [ ]:
# Fit KMeans with your chosen k on X_pca_blob
# Add the cluster labels as a new column 'cluster' in the df DataFrame
# Print the cluster sizes using value_counts()

# Your code here


In [ ]:
#@title Click to reveal solution
kmeans_b = KMeans(n_clusters=best_k_b, random_state=42, n_init=10)
df['cluster'] = kmeans_b.fit_predict(X_pca_blob)

print("Cluster sizes:")
print(df['cluster'].value_counts().sort_index())

### Task 5: Visualize Clusters in 2D PCA Space

In [ ]:
# Fit a separate PCA with n_components=2 for visualization
# Plot the data colored by cluster
# To plot centroids: inverse_transform them from the clustering PCA space
#   back to scaled space, then forward_transform into the 2D PCA space
#   centers_scaled = pca_b_reduced.inverse_transform(kmeans_b.cluster_centers_)
#   centers_2d = pca_2d_viz.transform(centers_scaled)

# Your code here


In [ ]:
#@title Click to reveal solution
pca_2d_viz = PCA(n_components=2, random_state=42)
X_2d_blob = pca_2d_viz.fit_transform(X_blob_scaled)

centers_scaled_b = pca_b_reduced.inverse_transform(kmeans_b.cluster_centers_)
centers_2d_b = pca_2d_viz.transform(centers_scaled_b)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_2d_blob[:, 0], X_2d_blob[:, 1], c=df['cluster'], cmap='tab10',
                     s=30, alpha=0.5)
ax.scatter(centers_2d_b[:, 0], centers_2d_b[:, 1], c='black', marker='X', s=250,
           edgecolors='white', linewidth=2, zorder=5, label='Centroids')
ax.set_xlabel(f'PC1 ({pca_2d_viz.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_2d_viz.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Clusters in 2D PCA Space (k={best_k_b})')
ax.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

### Task 6: Cluster Profile Heatmap

In [ ]:
# Compute mean feature values per cluster in the ORIGINAL scale
# Use: df.groupby('cluster')[feature_cols].mean()
# For a heatmap, standardize the means so features are visually comparable
# Display as an imshow heatmap with feature names on x-axis

# Your code here


In [ ]:
#@title Click to reveal solution
cluster_profile_orig = df.groupby('cluster')[feature_cols].mean()
print("Cluster profiles (original scale):")
print(cluster_profile_orig.round(2).to_string())

# Standardize for heatmap visualization
profile_std = cluster_profile_orig.copy()
for col in feature_cols:
    col_mean = df[col].mean()
    col_std = df[col].std()
    profile_std[col] = (profile_std[col] - col_mean) / col_std

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(profile_std.values, cmap='RdBu_r', aspect='auto', vmin=-2, vmax=2)
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right')
ax.set_yticks(range(best_k_b))
ax.set_yticklabels([f'Cluster {i}' for i in range(best_k_b)])
plt.colorbar(im, ax=ax, label='Relative Value (standardized)')
ax.set_title('Cluster Profiles')
plt.tight_layout()
plt.show()

### Task 7: Display Sample Rows from Each Cluster

In [ ]:
# For each cluster, display 3 random sample rows from the DataFrame
# Use: df[df['cluster'] == k].sample(3)

# Your code here


In [ ]:
#@title Click to reveal solution
for i in range(best_k_b):
    subset = df[df['cluster'] == i]
    print(f"--- Cluster {i} ({len(subset)} samples) ---")
    print(subset.sample(3, random_state=42)[feature_cols].round(2).to_string())
    print()

---
# Summary

**Key Takeaways:**

1. **The curse of dimensionality** makes distance-based methods unreliable in high dimensions. Dimensionality reduction is the remedy.

2. **PCA** finds orthogonal directions of maximum variance and projects your data onto them. It's fast, deterministic, and interpretable via loadings. Always standardize first.

3. **Choosing the number of components:** use scree plots, cumulative variance thresholds (90–95%), or cross-validation on a downstream task.

4. **t-SNE and UMAP** are powerful visualization tools that capture nonlinear structure, but they are not reliable for preprocessing. Distances between clusters in t-SNE are not meaningful, and results change with hyperparameters.

5. **PCA + K-Means** is a practical pipeline: scale → PCA → cluster → visualize → interpret. Compare to clustering without PCA as a baseline.

6. **Interpreting clusters after PCA** requires mapping back to original features via `inverse_transform()` or computing group means on the original data.

**For Project 3**, you'll apply this exact pipeline to Spotify audio features: standardize, determine k, cluster, reduce with PCA for visualization, and interpret your clusters as musical "vibes."